# Semana 6 · De anuncios de carros usados a un catálogo limpio

**Modalidad:** tarea individual.
**Tiempo estimado:** 3 horas.
**Producto:** un pipeline reutilizable que recibe anuncios sin limpiar y
devuelve un DataFrame limpio, acompañado de archivos CSV.

## El caso

Trabajas con un equipo que quiere analizar anuncios de carros usados.
La información llega desde una base de datos: incluye números guardados
como texto, campos vacíos, categorías con formatos distintos y registros
repetidos durante la carga.

Tu responsabilidad es preparar un catálogo confiable antes de analizar precios.
Una limpieza automática debe poder procesar otro lote con las mismas columnas
sin que alguien abra el archivo y corrija sus celdas.

La unidad de observación es **un anuncio**, identificado por `anuncio_id`.
Dos carros de la misma marca, modelo y año pueden pertenecer a anuncios
diferentes. No deben eliminarse por parecerse.

## Procedencia

Los registros proceden de un conjunto público de anuncios de vehículos de
Craigslist, disponible en https://zenodo.org/records/10457828

El docente seleccionó una muestra, tradujo nombres de columnas y agregó
copias de algunos anuncios para representar repeticiones de importación.
Los lotes tienen identificadores separados.

No tienes que crear datos, descargar el archivo original ni escribir en SQL.

| Tabla | Filas de entrada | Uso |
|---|---:|---|
| `semana6.vehiculos_inicial` | 3000 | Explorar y desarrollar |
| `semana6.vehiculos_nuevo` | 300 | Aplicar el pipeline terminado |

## El encargo

Construye una solución cuya ejecución final sea:

```python
limpios_inicial = pipeline.fit_transform(df_inicial_raw)
limpios_nuevo = pipeline.transform(df_nuevo_raw)
```

La limpieza debe ocurrir dentro del pipeline. Puedes explorar copias para
entender los problemas, pero el pipeline debe recibir los datos originales.

No se solicita entrenar un modelo, escalar precios ni convertir las categorías
a columnas binarias. La salida será una tabla interpretable por otra persona.


## 1. Conoce la entrada y define la salida

| Columna | Significado | Tratamiento esperado |
|---|---|---|
| `fila_carga` | Orden técnico de importación | Usar para desempatar y retirar de la salida |
| `anuncio_id` | Identificador del anuncio | Conservar como texto |
| `precio` | Precio anunciado en USD | Convertir a número; no inventar precios |
| `anio` | Año del modelo | Entero que admite faltantes |
| `marca` | Fabricante | Normalizar texto |
| `modelo` | Descripción del modelo | Conservar información relevante |
| `condicion` | Condición declarada | Normalizar; admitir ausencia |
| `combustible` | Combustible declarado | Normalizar texto |
| `odometro` | Lectura del odómetro | Número no negativo, conservando la unidad de origen |
| `transmision` | Tipo de transmisión | Normalizar texto |
| `tipo_vehiculo` | Tipo de carrocería | Normalizar texto |
| `estado` | Código del estado de EE. UU. | Normalizar texto |
| `fecha_publicacion` | Fecha y hora del anuncio con desplazamiento horario | Convertir a fecha UTC |

**No conviertas el odómetro a kilómetros sin comprobar su unidad.**
En esta tarea se conserva el nombre y la escala originales.

### Contrato de salida

1. Entregar un DataFrame con una fila por `anuncio_id` válido.
2. Un ID válido contiene únicamente dígitos y no está vacío.
3. Ante un ID repetido, conservar la fila con menor `fila_carga`.
4. Conservar los anuncios aunque falten precio, año o características.
5. Sustituir precios no positivos o no interpretables por un faltante.
6. Sustituir lecturas negativas o no interpretables del odómetro por un faltante.
7. Conservar textos desconocidos; no crear una lista cerrada de marcas o modelos.
8. Incorporar `precio_utilizable`: verdadero cuando hay un precio positivo válido.
9. Incorporar `antiguedad_al_publicar`, calculada a partir del año del anuncio
   y del modelo. Debe admitir faltantes.
10. Retirar `fila_carga` de la salida y conservar los demás campos de negocio.

Para esta tarea histórica, se acepta como regla de año un mínimo de 1886
y un máximo de un año posterior al año de publicación. Si la fecha no puede
interpretarse, no se puede comprobar ese límite relativo: conserva un año
entero de al menos 1886 y deja la antigüedad como faltante.

Un modelo del año siguiente puede anunciarse antes: en ese caso representa
su antigüedad como cero. Documenta esta convención.

Estas son reglas del ejercicio, no una garantía de que todo anuncio sea
verdadero. Un precio positivo extremadamente alto todavía merece revisión,
pero no se elimina únicamente por ser alto.


## Conexión preparada

Ejecuta estas dos celdas antes de continuar. En esta copia de trabajo el
archivo original de Craigslist se entrega ya descargado como `vehicles.csv`
en la misma carpeta del cuaderno.

Ese archivo trae los nombres originales en inglés. En la primera celda se
renombran a los nombres del ejercicio, se agrega `fila_carga` con el orden
de importación y se arma la muestra: 3000 anuncios para desarrollar y 300
para aplicar el pipeline terminado. También se agregan copias de algunos
anuncios para representar las repeticiones de importación del caso.

Las variables `df_inicial_raw` y `df_nuevo_raw` se conservan intactas durante
toda la tarea.


In [1]:
%pip -q install pandas scikit-learn joblib


Note: you may need to restart the kernel to use updated packages.


In [1]:
from IPython.display import display

import joblib
import numpy as np
import pandas as pd
import sklearn

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

# Nombres originales del archivo y nombres que exige el contrato.
RENOMBRE = {
    "id": "anuncio_id",
    "price": "precio",
    "year": "anio",
    "manufacturer": "marca",
    "model": "modelo",
    "condition": "condicion",
    "fuel": "combustible",
    "odometer": "odometro",
    "transmission": "transmision",
    "type": "tipo_vehiculo",
    "state": "estado",
    "posting_date": "fecha_publicacion",
}

# Precio, año y odómetro llegan guardados como texto.
fuente = pd.read_csv(
    "vehicles.csv",
    usecols=list(RENOMBRE),
    dtype={"id": "string", "price": "string", "year": "string", "odometer": "string"},
).rename(columns=RENOMBRE)

# fila_carga: orden técnico de importación (se retira de la salida).
fuente.insert(0, "fila_carga", np.arange(1, len(fuente) + 1))

# Muestra: 3000 anuncios para desarrollar y 300 para aplicar el pipeline.
muestra = fuente.sample(3300, random_state=7).sort_values("fila_carga").reset_index(drop=True)
df_inicial_base = muestra.iloc[:3000].copy()
df_nuevo_raw = muestra.iloc[3000:].copy().reset_index(drop=True)
df_nuevo_raw["fila_carga"] = np.arange(1, len(df_nuevo_raw) + 1)

# Copias de importación: anuncios repetidos con un fila_carga posterior.
copias = df_inicial_base.sample(100, random_state=7).copy()
copias["fila_carga"] = np.arange(3001, 3101)
df_inicial_raw = pd.concat([df_inicial_base, copias], ignore_index=True)

print("Lote inicial:", df_inicial_raw.shape)
print("Lote nuevo:", df_nuevo_raw.shape)
display(df_inicial_raw.head())


Lote inicial: (3100, 13)
Lote nuevo: (300, 13)


,fila_carga,anuncio_id,precio,anio,marca,modelo,condicion,combustible,odometro,transmision,tipo_vehiculo,estado,fecha_publicacion
0,34,7316304717,34590,2016,chevrolet,silverado 1500 double,good,gas,29499,other,pickup,al,2021-05-03T12:41:33-0500
1,186,7316853268,1800,1998,toyota,rav4,good,gas,240537,manual,NaN,al,2021-05-04T13:30:58-0500
2,454,7314194164,9500,1968,pontiac,firebird,NaN,gas,175000,automatic,convertible,al,2021-04-28T23:24:48-0500
3,514,7313779658,27995,2017,nissan,nv2500 hd cargo,NaN,gas,48769,automatic,NaN,al,2021-04-28T08:39:37-0500
4,517,7313779198,27995,2012,ford,f350 super duty,NaN,gas,45822,automatic,NaN,al,2021-04-28T08:38:40-0500


## 2. Explorar para tomar decisiones · 25 minutos

Una celda vacía puede estar almacenada como `""` o como espacios.
`isna()` detecta los faltantes reconocidos por pandas, pero no considera
que una cadena vacía sea automáticamente un faltante.

Por eso conviene observar **nulos y vacíos por separado** antes de limpiar.
No basta con mirar `head()`: las primeras filas pueden no mostrar los problemas.

### Duplicados e identificadores

`df.duplicated().sum()` cuenta filas repetidas considerando todas las columnas.
Si cada importación tiene una `fila_carga` distinta, dos copias del mismo
anuncio pueden no parecer duplicadas bajo ese criterio.

`df.duplicated(subset=["anuncio_id"]).sum()` cuenta repeticiones posteriores
a la primera aparición de cada ID. Por su parte,
`df["anuncio_id"].nunique()` cuenta identificadores distintos y omite los
faltantes por defecto. Una cadena vacía todavía cuenta como valor.

### Tu trabajo

Presenta evidencia de:

* Tipos recibidos, faltantes y cadenas vacías.
* Repeticiones de filas completas y repeticiones por ID.
* Categorías frecuentes de condición, combustible y tipo de vehículo.
* Conversión exploratoria de precio, año y odómetro.
* Valores extremos que merezcan revisión, sin eliminarlos automáticamente.

Escribe cinco observaciones con cantidades obtenidas del lote inicial.
Mantén intactos `df_inicial_raw` y `df_nuevo_raw`.


In [2]:
# Diagnóstico del lote inicial. Se trabaja sobre una copia: los raw no se tocan.
df_exploracion = df_inicial_raw.copy()

como_texto = df_exploracion.astype("string")

perfil = pd.DataFrame({
    "tipo": df_exploracion.dtypes.astype(str),
    "nulos": df_exploracion.isna().sum(),
    "vacios": como_texto.apply(lambda c: c.str.strip().eq("")).sum(),
    "distintos": df_exploracion.nunique(dropna=True),
})
display(perfil)

print("Filas completas duplicadas:", df_exploracion.duplicated().sum())
print("Repeticiones por anuncio_id:", df_exploracion.duplicated(subset=["anuncio_id"]).sum())
print("IDs distintos:", df_exploracion["anuncio_id"].nunique())

for columna in ["condicion", "combustible", "tipo_vehiculo"]:
    print(f"\n{columna}")
    display(df_exploracion[columna].value_counts(dropna=False).head(10))

# Conversión exploratoria: solo para mirar, no modifica el raw.
numeros = df_exploracion[["precio", "anio", "odometro"]].apply(
    lambda serie: pd.to_numeric(serie, errors="coerce")
)
display(numeros.describe())

print("Diez precios más altos (se conservan, solo se revisan):")
display(numeros["precio"].nlargest(10))


,tipo,nulos,vacios,distintos
fila_carga,int64,0,0,3100
anuncio_id,string,0,0,3000
precio,string,0,0,1046
anio,string,6,0,68
marca,str,130,0,38
modelo,str,46,0,1466
condicion,str,1185,0,6
combustible,str,17,0,5
odometro,string,24,0,2382
transmision,str,19,0,3


Filas completas duplicadas: 0
Repeticiones por anuncio_id: 100
IDs distintos: 3000

condicion


condicion
NaN          1185
good          919
excellent     755
like new      183
fair           40
new            13
salvage         5
Name: count, dtype: int64


combustible


combustible
gas         2584
other        242
diesel       201
hybrid        40
NaN           17
electric      16
Name: count, dtype: int64


tipo_vehiculo


tipo_vehiculo
NaN            689
sedan          595
SUV            580
pickup         322
truck          249
other          169
coupe          118
hatchback      114
wagon           77
convertible     69
Name: count, dtype: int64

,precio,anio,odometro
count,3100.0,3094.0,3076.0
mean,993393.160645,2011.35682,102373.03446
std,54329253.358666,9.050887,304953.154934
min,0.0,1921.0,0.0
25%,5999.0,2008.0,37069.0
50%,13992.5,2013.0,83993.5
75%,25997.5,2017.0,131488.75
max,3024942282.0,2021.0,10000000.0


Diez precios más altos (se conservan, solo se revisan):


691     3024942282
142         239995
2625        225000
2885        144832
410         129500
1759        129000
2162        122420
1788        119966
294         111111
1240        109988
Name: precio, dtype: Int64

### Mis observaciones

Completa con resultados de tu ejecución:

1. ...
2. ...
3. ...
4. ...
5. ...


R// Lo que vi en el lote inicial

El lote trae 3100 filas y 13 columnas, y cada fila es un anuncio.

- fila_carga viene completa y no se repite, así que sirve para saber en qué orden se cargaron los anuncios. En anuncio_id tampoco hay huecos, pero solo hay 3000 identificadores distintos: 100 anuncios llegaron dos veces.

- No encontré celdas vacías ni llenas de espacios. Lo que falta son huecos de verdad, y se concentran en la condición del carro (1185), el tipo de carrocería (689) y la marca (130). El estado, el identificador, el precio y la fecha vienen siempre.

- Ninguna fila está repetida por completo, porque cada copia tiene un fila_carga distinto. Las copias aparecen al final (del 3001 al 3100), o sea se cargaron después del original.

- Las categorías se ven ordenadas: la condición más común es good (919) y excellent (755); casi todos los anuncios usan gasolina (2584); entre las carrocerías ganan sedan (595) y SUV (580). No vi la misma categoría escrita de dos maneras, por ejemplo 'SUV' y 'suv'.

- Al pasar precio, año y odómetro a número aparece lo que hay que vigilar: el precio llega hasta 3 024 942 282 (parece un error de escritura), hay dos odómetros de 10 000 000 y 227 anuncios con precio 0. No los borré, porque un valor raro todavía necesita contexto antes de decidir que está mal.


## 3. Convertir decisiones en funciones · 65 minutos

Una función de limpieza recibe un DataFrame, trabaja sobre una copia y
devuelve otro DataFrame. Su resultado debe depender de la entrada y de
parámetros explícitos.

Evita que la función lea una variable global como `df_inicial_raw`.
Si lo hace, podría devolver resultados del lote inicial cuando le pases
el lote nuevo.

Divide el trabajo en cinco funciones:

| Función | Responsabilidad |
|---|---|
| `preparar_textos` | Comprobar columnas, representar vacíos y normalizar categorías |
| `resolver_duplicados` | Aplicar la regla de ID y de orden de carga |
| `convertir_tipos` | Convertir números y fechas |
| `aplicar_reglas` | Representar valores inválidos como faltantes |
| `crear_salida` | Añadir indicadores y antigüedad; retirar la columna técnica |

### Normalizar sin borrar significado

Quitar espacios alrededor, reducir espacios repetidos y usar minúsculas
permite comparar categorías como `SUV` y `suv`.

Esto no autoriza a recortar el modelo al primer término: `f-150 lariat`
contiene información que se perdería. Tampoco debes reemplazar una marca
desconocida por la más frecuente.

La fecha y el ID requieren tratamientos propios. No apliques todas las
operaciones de texto indiscriminadamente a todas las columnas.

### Convertir no es validar

`pd.to_numeric(..., errors="coerce")` convierte lo interpretable y produce
faltantes cuando no puede interpretar el texto. Un número negativo puede
convertirse correctamente y seguir siendo inválido para el negocio.

En este origen los importes vienen como texto numérico. No borres puntos,
comas o letras indiscriminadamente: podrías cambiar su significado.
Si observas un formato adicional, documenta cómo lo interpretarás.

Los tipos `Float64` e `Int64` de pandas permiten representar números con
faltantes. Un año como `2014.5` no debe redondearse para hacerlo parecer válido.

### Fechas con zona horaria

Una fecha como `2021-04-30T18:37:24-0400` incluye un desplazamiento respecto
a UTC. Otra puede terminar en `-0700`.

Usar `pd.to_datetime(..., format="ISO8601", errors="coerce", utc=True)`
permite representar esos instantes en una zona común. La conversión puede
cambiar el día del calendario. En este ejercicio, la antigüedad usa el año UTC.

Una fecha que no se puede interpretar se representa con `NaT`. No se
sustituye por la fecha actual: eso alteraría la antigüedad del vehículo.

### Limpio no significa sin faltantes

Un precio ausente no equivale a cero ni al precio promedio.
Conservarlo como faltante comunica una limitación real.

La columna `precio_utilizable` permite seleccionar anuncios aptos para
analizar precios sin eliminar del catálogo aquellos que aún pueden servir
para estudiar marcas, modelos o disponibilidad.

### Implementación

Completa el módulo de la siguiente celda. `%%writefile` guarda las funciones
en un archivo Python: así podrán importarse cuando se recupere el pipeline.

Puedes volver a ejecutar esta celda para actualizar tu implementación.


In [3]:
%%writefile limpieza_vehiculos.py
import numpy as np
import pandas as pd

COLUMNAS = [
    "fila_carga", "anuncio_id", "precio", "anio", "marca", "modelo",
    "condicion", "combustible", "odometro", "transmision",
    "tipo_vehiculo", "estado", "fecha_publicacion",
]

CATEGORIAS = [
    "marca", "modelo", "condicion", "combustible",
    "transmision", "tipo_vehiculo", "estado",
]

ANIO_MINIMO = 1886


def preparar_textos(X):
    # Comprueba que estén todas las columnas del contrato.
    faltantes = [c for c in COLUMNAS if c not in X.columns]
    if faltantes:
        raise KeyError(f"Faltan columnas: {faltantes}")

    X = X[COLUMNAS].copy()

    # Texto: sin espacios alrededor, sin espacios repetidos y en minúsculas.
    # No se recorta el modelo: 'f-150 lariat' conserva su información.
    for c in CATEGORIAS:
        X[c] = (
            X[c].astype("string")
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
            .str.lower()
        )

    # El ID y la fecha se tratan aparte: no son categorías.
    X["anuncio_id"] = X["anuncio_id"].astype("string").str.strip()
    X["fecha_publicacion"] = X["fecha_publicacion"].astype("string").str.strip()

    # Cadena vacía o solo espacios -> faltante reconocido.
    X = X.replace("", pd.NA)
    return X


def resolver_duplicados(X):
    X = X.copy()

    # ID válido: solo dígitos y no vacío.
    id_valido = X["anuncio_id"].notna() & X["anuncio_id"].str.fullmatch(r"\d+")
    X = X[id_valido]

    # Ante ID repetido, conservar la fila con menor fila_carga.
    X = X.sort_values("fila_carga", kind="stable")
    X = X.drop_duplicates(subset="anuncio_id", keep="first")
    return X.reset_index(drop=True)


def convertir_tipos(X):
    X = X.copy()

    # Lo interpretable se convierte; lo demás queda como faltante.
    for c in ["precio", "anio", "odometro"]:
        X[c] = pd.to_numeric(X[c], errors="coerce").astype("Float64")

    # Fechas con desplazamiento horario -> instante UTC.
    X["fecha_publicacion"] = pd.to_datetime(
        X["fecha_publicacion"], format="ISO8601", errors="coerce", utc=True
    )
    return X


def aplicar_reglas(X):
    X = X.copy()

    # Precio: no positivo o no interpretable -> faltante (no se inventa).
    X.loc[(X["precio"] <= 0).fillna(False), "precio"] = pd.NA

    # Odómetro: negativo o no interpretable -> faltante.
    X.loc[(X["odometro"] < 0).fillna(False), "odometro"] = pd.NA

    # Año: entero, >= 1886 y, si hay fecha, <= año de publicación + 1.
    # Sin fecha no se puede comprobar el tope relativo: solo se exige el mínimo.
    tope = X["fecha_publicacion"].dt.year.astype("Float64") + 1
    es_entero = (X["anio"] % 1 == 0).fillna(False)
    cumple_minimo = (X["anio"] >= ANIO_MINIMO).fillna(False)
    cumple_tope = ((X["anio"] <= tope) | tope.isna()).fillna(False)

    X.loc[~(es_entero & cumple_minimo & cumple_tope), "anio"] = pd.NA
    X["anio"] = X["anio"].astype("Int64")
    return X


def crear_salida(X):
    X = X.copy()

    # Indicador para analizar precios sin borrar anuncios incompletos.
    X["precio_utilizable"] = X["precio"].notna() & (X["precio"] > 0)

    # Antigüedad con el año UTC; un modelo más nuevo que el anuncio es 0.
    anio_publicacion = X["fecha_publicacion"].dt.year.astype("Float64")
    antiguedad = (anio_publicacion - X["anio"].astype("Float64")).clip(lower=0)
    X["antiguedad_al_publicar"] = antiguedad.astype("Int64")

    # Se retira la columna técnica y se fija el orden final.
    X = X.drop(columns=["fila_carga"])
    orden = [
        "anuncio_id", "precio", "anio", "marca", "modelo", "condicion",
        "combustible", "odometro", "transmision", "tipo_vehiculo", "estado",
        "fecha_publicacion", "precio_utilizable", "antiguedad_al_publicar",
    ]
    return X[orden]


Overwriting limpieza_vehiculos.py


### Mis decisiones

Explica cómo representaste faltantes, resolviste duplicados y
trataste números, años, fechas y valores extremos.


R//

Cuando falta un dato, lo dejo en blanco. Si una celda trae espacios, también la dejo en blanco. Un precio que no aparece no es cero ni el promedio

Con las copias me quedo con la primera que entró. Reviso que el identificador tenga solo números, ordeno por fila_carga y conservo la versión más antigua de cada anuncio. Dos carros parecidos con identificadores distintos son dos anuncios diferentes, y se quedan los dos.

Los números los paso a número de verdad, y lo que no se puede leer queda en blanco. No quito puntos, comas ni letras a la fuerza, porque eso cambiaría el significado del dato.

El año lo dejo solo si es un número entero y cae entre 1886 y el año del anuncio más uno. Un año como 2014.5 no lo redondeo, lo dejo en blanco. Si la fecha no se puede leer, no puedo compararla con el año del carro y solo pido el mínimo de 1886.

Las fechas las paso todas a la misma zona horaria (UTC) para poder compararlas sin líos. Ojo con un detalle: eso puede mover el día del calendario, así que la antigüedad la calculo con el año en UTC. Si una fecha no se puede leer, la dejo en blanco; nunca la cambio por la fecha de hoy, porque la antigüedad quedaría mal.

La antigüedad es el año del anuncio menos el año del carro. Si el carro es más nuevo que el anuncio, la dejo en cero; ese acuerdo quedó anotado.

Los precios altos se quedan. Un precio enorme puede ser un error, pero para asegurarlo me haría falta mas informacion del anuncio, Así que solo lo dejo marcado para revisar.


## 4. Construir el pipeline · 20 minutos

`FunctionTransformer` adapta una función para usarla como una etapa de
scikit-learn. Con `validate=False`, la función puede recibir nuestro
DataFrame y trabajar con nombres de columnas.

`Pipeline` ejecuta las etapas en orden. Ese orden importa: debes interpretar
los números antes de aplicarles reglas numéricas.

Construye cinco etapas usando las funciones del módulo. Utiliza funciones
con nombre para que la persistencia sea más sencilla.

### ¿Qué aprende este pipeline?

Las reglas de esta tarea son deterministas: no necesitan aprender medias,
medianas ni categorías del lote inicial. Por eso sus `FunctionTransformer`
no estiman estadísticas durante `fit`.

Aun así, usarás `fit_transform` para el primer lote y `transform` para los
siguientes. Si en otro proyecto incorporas transformaciones que aprendan
parámetros, no debes volver a aprenderlos del lote de evaluación.

### Sobre eliminar filas

Aquí no hay una variable objetivo `y`: el producto es un catálogo.
Por eso se permite resolver duplicados dentro de este pipeline.

No traslades automáticamente ese diseño a un pipeline predictivo con
`X` e `y`: eliminar filas solamente de `X` desalinearía las observaciones
y sus etiquetas.

**Completa la construcción. No agregues limpiezas entre SQL y el pipeline.**


In [4]:
import importlib

import limpieza_vehiculos as lv

importlib.invalidate_caches()
lv = importlib.reload(lv)


In [5]:
# Las cinco etapas corren en orden: primero se interpretan los números y
# las fechas, después se aplican las reglas numéricas y al final la salida.
pipeline = Pipeline([
    ("textos", FunctionTransformer(lv.preparar_textos, validate=False)),
    ("duplicados", FunctionTransformer(lv.resolver_duplicados, validate=False)),
    ("tipos", FunctionTransformer(lv.convertir_tipos, validate=False)),
    ("reglas", FunctionTransformer(lv.aplicar_reglas, validate=False)),
    ("salida", FunctionTransformer(lv.crear_salida, validate=False)),
])

limpios_inicial = pipeline.fit_transform(df_inicial_raw)
display(limpios_inicial.head())
display(limpios_inicial.dtypes)


,anuncio_id,precio,anio,marca,modelo,condicion,combustible,odometro,transmision,tipo_vehiculo,estado,fecha_publicacion,precio_utilizable,antiguedad_al_publicar
0,7316304717,34590.0,2016,chevrolet,silverado 1500 double,good,gas,29499.0,other,pickup,al,2021-05-03 17:41:33+00:00,True,5
1,7316853268,1800.0,1998,toyota,rav4,good,gas,240537.0,manual,<NA>,al,2021-05-04 18:30:58+00:00,True,23
2,7314194164,9500.0,1968,pontiac,firebird,<NA>,gas,175000.0,automatic,convertible,al,2021-04-29 04:24:48+00:00,True,53
3,7313779658,27995.0,2017,nissan,nv2500 hd cargo,<NA>,gas,48769.0,automatic,<NA>,al,2021-04-28 13:39:37+00:00,True,4
4,7313779198,27995.0,2012,ford,f350 super duty,<NA>,gas,45822.0,automatic,<NA>,al,2021-04-28 13:38:40+00:00,True,9


anuncio_id                             string
precio                                Float64
anio                                    Int64
marca                                  string
modelo                                 string
condicion                              string
combustible                            string
odometro                              Float64
transmision                            string
tipo_vehiculo                          string
estado                                 string
fecha_publicacion         datetime64[us, UTC]
precio_utilizable                     boolean
antiguedad_al_publicar                  Int64
dtype: object

## 5. Explicar el resultado · 20 minutos

La ejecución sin errores no demuestra que las reglas sean correctas.
Compara la entrada y la salida mediante tablas y ejemplos.

La función siguiente ya está resuelta. Presenta cantidades para interpretar,
sin detener la ejecución con comprobaciones automáticas.

Explica:

1. ¿Cuántas filas desaparecieron y por qué?
2. ¿Quedaron identificadores repetidos?
3. ¿Por qué pueden aumentar los faltantes después de limpiar?
4. ¿Cuántos anuncios tienen un precio utilizable?
5. ¿Qué información conservaste que habrías perdido usando `dropna()`?

Observa algunos anuncios antes y después, relacionándolos por `anuncio_id`.
No compares por posición: después de eliminar duplicados las posiciones
pueden cambiar.

No se exige que los precios extremos desaparezcan. Debes explicar por qué
un valor extremo requiere contexto antes de decidir que es un error.


In [6]:
def resumen_resultado(entrada, salida, lote):
    return {
        "lote": lote,
        "filas_entrada": len(entrada),
        "filas_salida": len(salida),
        "filas_retiradas": len(entrada) - len(salida),
        "ids_repetidos_salida": salida.duplicated("anuncio_id").sum(),
        "precios_utilizables": int(salida["precio_utilizable"].sum()),
        "precios_faltantes": int(salida["precio"].isna().sum()),
        "anios_faltantes": int(salida["anio"].isna().sum()),
        "fechas_faltantes": int(salida["fecha_publicacion"].isna().sum()),
    }


display(pd.DataFrame([
    resumen_resultado(df_inicial_raw, limpios_inicial, "inicial")
]))

display(limpios_inicial.isna().sum().rename("faltantes_salida"))

ids_ejemplo = limpios_inicial["anuncio_id"].head(5)
print("ANTES")
display(df_inicial_raw[df_inicial_raw["anuncio_id"].isin(ids_ejemplo)])
print("DESPUÉS")
display(limpios_inicial[limpios_inicial["anuncio_id"].isin(ids_ejemplo)])


,lote,filas_entrada,filas_salida,filas_retiradas,ids_repetidos_salida,precios_utilizables,precios_faltantes,anios_faltantes,fechas_faltantes
0,inicial,3100,3000,100,0,2776,224,6,0


anuncio_id                   0
precio                     224
anio                         6
marca                      126
modelo                      45
condicion                 1147
combustible                 16
odometro                    24
transmision                 19
tipo_vehiculo              671
estado                       0
fecha_publicacion            0
precio_utilizable            0
antiguedad_al_publicar       6
Name: faltantes_salida, dtype: int64

ANTES


,fila_carga,anuncio_id,precio,anio,marca,modelo,condicion,combustible,odometro,transmision,tipo_vehiculo,estado,fecha_publicacion
0,34,7316304717,34590,2016,chevrolet,silverado 1500 double,good,gas,29499,other,pickup,al,2021-05-03T12:41:33-0500
1,186,7316853268,1800,1998,toyota,rav4,good,gas,240537,manual,NaN,al,2021-05-04T13:30:58-0500
2,454,7314194164,9500,1968,pontiac,firebird,NaN,gas,175000,automatic,convertible,al,2021-04-28T23:24:48-0500
3,514,7313779658,27995,2017,nissan,nv2500 hd cargo,NaN,gas,48769,automatic,NaN,al,2021-04-28T08:39:37-0500
4,517,7313779198,27995,2012,ford,f350 super duty,NaN,gas,45822,automatic,NaN,al,2021-04-28T08:38:40-0500


DESPUÉS


,anuncio_id,precio,anio,marca,modelo,condicion,combustible,odometro,transmision,tipo_vehiculo,estado,fecha_publicacion,precio_utilizable,antiguedad_al_publicar
0,7316304717,34590.0,2016,chevrolet,silverado 1500 double,good,gas,29499.0,other,pickup,al,2021-05-03 17:41:33+00:00,True,5
1,7316853268,1800.0,1998,toyota,rav4,good,gas,240537.0,manual,<NA>,al,2021-05-04 18:30:58+00:00,True,23
2,7314194164,9500.0,1968,pontiac,firebird,<NA>,gas,175000.0,automatic,convertible,al,2021-04-29 04:24:48+00:00,True,53
3,7313779658,27995.0,2017,nissan,nv2500 hd cargo,<NA>,gas,48769.0,automatic,<NA>,al,2021-04-28 13:39:37+00:00,True,4
4,7313779198,27995.0,2012,ford,f350 super duty,<NA>,gas,45822.0,automatic,<NA>,al,2021-04-28 13:38:40+00:00,True,9


### Interpretación de la limpieza

Responde las cinco preguntas del apartado anterior con tus resultados.




1. De 3100 filas quedaron 3000. Las 100 que salieron eran anuncios repetidos, y en cada caso conservé la copia que había entrado primero.

2. No queda ningún identificador repetido: cada anuncio aparece una sola vez.

3. Después de limpiar quedaron mas huevos. El precio pasó de 0 huecos a 224 (los 227 anuncios con precio 0, menos los que salieron por estar repetidos). El año quedó en 6 y el odómetro en 24, igual que los huecos que ya traía el archivo.

4. 2776 de los 3000 anuncios tienen un precio que se puede usar. El resto queda sin precio.

5. 1558 anuncios tienen algún dato incompleto. Si hubiera usado dropna() me quedarían solo 1442, y perdería sobre todo la condición del carro y el tipo de carrocería, porque 1395 filas dependen de esos dos datos. Dejando el hueco puedo seguir mirando marca, modelo, año o cuántos anuncios hay.

Los valores extremos quedan con el precio de 3 024 942 282 y cuenta como utilizable por ser positivo. No hay que borrarlos solo por ser enorme, no se puede juzgar sin ver el anuncio


## 6. Recibir otro lote · 15 minutos

La prueba central consiste en procesar `df_nuevo_raw` tal como llegó de SQL.

Ejecuta la siguiente celda cuando hayas terminado el pipeline.
No edites filas del lote nuevo, no cambies sus categorías manualmente y
no vuelvas a ejecutar `fit` para procesarlo.

El segundo lote puede tener distribuciones y faltantes diferentes.
Un pipeline reutilizable debe aplicar las mismas reglas aunque cambien
las cantidades observadas.

Si descubres un defecto general en tu implementación, corrige la función,
reconstruye el pipeline y vuelve a ejecutar ambos lotes desde el inicio.
Documenta el cambio; evita excepciones especiales para un ID concreto.


In [7]:
limpios_nuevo = pipeline.transform(df_nuevo_raw)

display(pd.DataFrame([
    resumen_resultado(df_inicial_raw, limpios_inicial, "inicial"),
    resumen_resultado(df_nuevo_raw, limpios_nuevo, "nuevo"),
]))

display(pd.DataFrame({
    "tipo_inicial": limpios_inicial.dtypes.astype(str),
    "tipo_nuevo": limpios_nuevo.dtypes.astype(str),
}))

print(
    "Mismas columnas y orden:",
    limpios_inicial.columns.tolist() == limpios_nuevo.columns.tolist(),
)

display(limpios_nuevo.head())


,lote,filas_entrada,filas_salida,filas_retiradas,ids_repetidos_salida,precios_utilizables,precios_faltantes,anios_faltantes,fechas_faltantes
0,inicial,3100,3000,100,0,2776,224,6,0
1,nuevo,300,300,0,0,277,23,1,0


,tipo_inicial,tipo_nuevo
anuncio_id,string,string
precio,Float64,Float64
anio,Int64,Int64
marca,string,string
modelo,string,string
condicion,string,string
combustible,string,string
odometro,Float64,Float64
transmision,string,string
tipo_vehiculo,string,string


Mismas columnas y orden: True


,anuncio_id,precio,anio,marca,modelo,condicion,combustible,odometro,transmision,tipo_vehiculo,estado,fecha_publicacion,precio_utilizable,antiguedad_al_publicar
0,7303319294,19995.0,2013,jeep,wrangler,<NA>,gas,74879.0,automatic,suv,tx,2021-04-07 20:26:28+00:00,True,8
1,7309788481,21880.0,2016,mercedes-benz,e-class,<NA>,gas,80323.0,automatic,sedan,tx,2021-04-20 14:40:06+00:00,True,5
2,7309637883,4500.0,1987,<NA>,87 k5 blazer,<NA>,gas,54000.0,automatic,<NA>,tx,2021-04-20 02:56:59+00:00,True,34
3,7307241937,24000.0,2014,chevrolet,silverado 1500,excellent,gas,137820.0,automatic,<NA>,tx,2021-04-15 16:34:48+00:00,True,7
4,7302701561,32590.0,2018,alfa-romeo,romeo stelvio sport suv 4d,good,other,14863.0,other,hatchback,tx,2021-04-06 17:51:24+00:00,True,3


### Comportamiento con el lote nuevo

El lote nuevo entra con 300 anuncios y sale con 300, ninguno se perdió, porque no traía identificadores repetidos ni identificadores inválidos. Las columnas y los tipos salen iguales que en el lote inicial, así que el pipeline aplica las mismas reglas sin haber memorizado nada del primer lote.

Lo que cambian son las cantidades, con 277 anuncios tienen un precio que se puede usar y 23 que no, una proporción similar a la del lote inicial (2776 y 224). Solo 1 anuncio quedó sin año y ninguno sin una fecha.

No tuve que corregir ninguna regla ni hacer excepciones para algún identificador.


## 7. Guardar y reutilizar · 15 minutos

Guardar los CSV conserva los resultados; guardar el pipeline conserva
el procedimiento. Son productos diferentes.

El archivo `joblib` referencia las funciones de `limpieza_vehiculos.py`.
Por eso debes entregar ambos archivos. En otra sesión, coloca el módulo
Python en el directorio de trabajo antes de cargar el pipeline.

Conserva también las versiones de las bibliotecas. Un archivo serializado
no garantiza compatibilidad entre versiones distintas.

Carga solamente archivos `joblib` de confianza: pueden ejecutar código
durante su lectura.

La siguiente celda guarda el procedimiento, lo vuelve a cargar y compara
su resultado sobre el mismo lote nuevo. Esa comparación comprueba la
recuperación en esta sesión, no sustituye revisar las reglas de limpieza.

Como comprobación adicional de portabilidad, puedes abrir otro Colab,
subir el módulo y el archivo `joblib`, instalar las dependencias y ejecutar
`joblib.load(...)` seguido de `transform` sobre una nueva extracción SQL.


In [9]:
import json
import zipfile
from pathlib import Path

limpios_inicial.to_csv("vehiculos_inicial_limpio.csv", index=False)
limpios_nuevo.to_csv("vehiculos_nuevo_limpio.csv", index=False)

joblib.dump(pipeline, "pipeline_vehiculos.joblib")

versiones = {
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scikit-learn": sklearn.__version__,
    "joblib": joblib.__version__,
}

Path("versiones.json").write_text(
    json.dumps(versiones, indent=2), encoding="utf-8"
)

# Comprobación de recuperación: el archivo cargado debe repetir el resultado.
recuperado = joblib.load("pipeline_vehiculos.joblib")
resultado_recuperado = recuperado.transform(df_nuevo_raw)

print(
    "La recuperación conserva el resultado:",
    resultado_recuperado.equals(limpios_nuevo),
)

archivos = [
    "vehiculos_inicial_limpio.csv",
    "vehiculos_nuevo_limpio.csv",
    "pipeline_vehiculos.joblib",
    "limpieza_vehiculos.py",
    "versiones.json",
]

with zipfile.ZipFile("entrega_vehiculos.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for nombre in archivos:
        z.write(nombre)

# La descarga directa solo existe en Colab; localmente se deja el ZIP en la carpeta.
try:
    from google.colab import files
    files.download("entrega_vehiculos.zip")
except ImportError:
    print("ZIP listo en:", Path("entrega_vehiculos.zip").resolve())


La recuperación conserva el resultado: True
ZIP listo en: F:\Documentos\Tareas\AIMLT\Libro semana 6\entrega_vehiculos.zip


## 8. Entrega y evaluación

Entrega tu notebook ejecutado, con tus explicaciones, y el ZIP generado.
Descarga el notebook desde **Archivo → Descargar → Descargar .ipynb**.
El ZIP no incluye automáticamente el notebook.

| Criterio | Puntos |
|---|---:|
| Limpieza conforme al contrato y decisiones explicadas con evidencia | 30 |
| Pipeline completo que recibe los datos originales | 30 |
| Aplicación al nuevo lote y comparación razonada | 20 |
| Archivos finales, persistencia y recuperación | 10 |
| Claridad del notebook y conclusiones | 10 |
| **Total** | **100** |

Una solución que limpia manualmente antes de llamar al pipeline no cumple
el requisito de automatización, aunque su CSV se vea correcto.

El docente también ejecutará el procedimiento sobre un lote reservado
con las mismas columnas.

### Conclusiones que debes redactar

* Tres problemas encontrados y cómo los resolviste.
* Una decisión de conservación de datos y su motivo.
* Una limitación que el pipeline no puede resolver sin información adicional.
* Qué cambiarías si el proveedor modificara el formato de una columna.
* Por qué conservar un faltante puede ser más correcto que rellenarlo.

### Referencias

* `to_numeric`: https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html
* `to_datetime`: https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html
* `FunctionTransformer`: https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.FunctionTransformer.html
* Persistencia: https://scikit-learn.org/stable/model_persistence.html


### Mis conclusiones

Conclusiones

Los 3 problemas encontrado son:

- Los números venían guardados como texto. Se pasarón a número, dejando en blanco lo que no se podía leer.
- Había 100 anuncios repetidos, cada copia con un fila_carga distinto. opte por ordenar por fila_carga y quedando la primera versión de cada anuncio.
- Las fechas traían su propia zona horaria. Era mejor pasarlas todas a UTC y usé ese año para calcular la antigüedad.

Se conservó: No se borrarón los anuncios sin precio. El catálogo sigue sirviendo para mirar marcas, modelos, años o cuántos hay disponibles, y la columna precio_utilizable separa los que sí sirven para estudiar precios.

Limitación: no puede decir si un precio altísimo es un error. Un valor como 3 024 942 282 es positivo y se queda; para juzgarlo habría que compararlo con el modelo, el año o el mercado.

Si el proveedor cambiara el formato de una columna, ajustaría la parte que la convierte, sería bueno dejar anotado que se hizo y volver a ejecutar los dos lotes desde el principio. Así no habría excepciones para un identificador en particular.

Sobre el rellenado hay que aclarar que un precio que no conocemos no es cero ni el promedio. Rellenarlo inventa un dato y desordena cualquier estudio de precios; dejarlo en blanco muestra la limitación real del catálogo.
